<a href="https://colab.research.google.com/github/Pranav03125/LaundryWateWaterTreatment_ML/blob/main/EC2ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Step 1: Data Loading and Initial Overview
import pandas as pd

# Load the cleaned dataset (CSV version for ease in Colab)
file_path = '/content/Data_EF Laundry (1).xlsx'  # Update path if needed
df = pd.read_excel(file_path)

# Display first 5 rows to understand structure
print("First 5 rows of the dataset:")
print(df.head())

# Dataset shape and column info
print("\nDataset shape (rows, columns):", df.shape)
print("\nColumn names:")
print(df.columns)

# Data types and non-null counts
print("\nDataset info:")
df.info()

# Check for missing values count per column
print("\nMissing values count per column:")
print(df.isnull().sum())

# Basic descriptive statistics for numeric columns
print("\nDescriptive statistics summary:")
print(df.describe())


FileNotFoundError: [Errno 2] No such file or directory: '/content/Data_EF Laundry (1).xlsx'

In [ ]:
# Step 2: Data Cleaning and Preprocessing

from sklearn.impute import SimpleImputer

# Drop the column "Aluminium (LAS)" as per project requirements, if present
if 'Aluminium (LAS)' in df.columns:
    df = df.drop(columns=['Aluminium (LAS)'])

# If 'flowrate' is a string with units like "1.88L/min", convert it to float
if df['flowrate'].dtype == object:
    df['flowrate'] = df['flowrate'].str.replace('L/min', '').astype(float)

# Check missing values before imputation
print("Missing values before imputation:")
print(df.isnull().sum())

# List of columns with missing values to impute
num_cols_with_missing = ['LAS Absorbance', 'LAS', 'pH', 'Current', 'CD', 'Power']

# Impute missing numeric values with median
imputer = SimpleImputer(strategy='median')
df[num_cols_with_missing] = imputer.fit_transform(df[num_cols_with_missing])

# Encode 'Set' column as categorical type for modeling
df['Set'] = df['Set'].astype('category')

# Confirm no missing values remain
print("\nMissing values after imputation:")
print(df.isnull().sum())

# Show updated data types and preview data
print("\nData types after cleaning:")
print(df.dtypes)
print("\nData preview after cleaning:")
print(df.head())


In [ ]:
# Step 3: Exploratory Data Analysis (EDA)

import matplotlib.pyplot as plt
import seaborn as sns

# 1. Distribution plots for key numeric features and targets
key_vars = ['flowrate', 'Time', 'Current', 'CD', 'pH', 'Turbidity', 'LAS', 'COD']

plt.figure(figsize=(15, 12))
for i, var in enumerate(key_vars, 1):
    plt.subplot(3, 3, i)
    sns.histplot(df[var], kde=True, bins=20)
    plt.title(f'Distribution of {var}')
plt.tight_layout()
plt.show()

# 2. Correlation matrix heatmap including targets and key features
plt.figure(figsize=(12, 8))
corr_matrix = df[['flowrate', 'Time', 'Current', 'CD', 'pH', 'Turbidity', 'LAS', 'COD']].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix')
plt.show()

# 3. Scatter plots of LAS and COD vs Time and Current Density (CD) to check for trends
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
sns.scatterplot(x='Time', y='LAS', data=df, hue='Set', palette='deep', s=60)
plt.title('LAS vs Time by Set')

plt.subplot(1, 2, 2)
sns.scatterplot(x='CD', y='COD', data=df, hue='Set', palette='deep', s=60)
plt.title('COD vs Current Density by Set')

plt.tight_layout()
plt.show()


In [ ]:
# Step 4: Feature selection/engineering and train-test split

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Keep CD (drop Current and Volatge, ESA), drop LAS Absorbance if treating as proxy of LAS (optional toggle)
drop_cols = ['Volatge', 'ESA cm2', 'Current']
# If you want a pure process-input model (no lab proxies), also drop 'LAS Absorbance'
drop_lab_proxy = True
if drop_lab_proxy and 'LAS Absorbance' in df.columns:
    drop_cols.append('LAS Absorbance')

df_model = df.drop(columns=drop_cols)

# Categorical: One-hot encode 'Set'
df_model = pd.get_dummies(df_model, columns=['Set'], drop_first=True)

# Features and targets
target_cols = ['LAS', 'COD']
X = df_model.drop(columns=target_cols)
y = df_model[target_cols]

# Train-test split (stratify on nothing due to regression, keep random_state for reproducibility)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X shape:", X.shape, "y shape:", y.shape)
print("Train:", X_train.shape, "Test:", X_test.shape)
print("Final feature columns:", list(X.columns))


In [ ]:
# Step 5: Model Training and Evaluation

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.multioutput import MultiOutputRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np # Import numpy

# Numeric columns already encoded; for ridge, scale inputs
num_cols = X_train.columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[('num', StandardScaler(), num_cols)],
    remainder='drop'
)

def eval_model(name, model, Xtr, Xte, ytr, yte):
    ytr_pred = model.predict(Xtr)
    yte_pred = model.predict(Xte)
    def metrics(y_true, y_pred):
        mae = mean_absolute_error(y_true, y_pred, multioutput='raw_values')
        # Calculate RMSE by taking the square root of the mean squared error
        rmse = np.sqrt(mean_squared_error(y_true, y_pred, multioutput='raw_values'))
        r2 = r2_score(y_true, y_pred, multioutput='raw_values')
        return mae, rmse, r2
    tr_mae, tr_rmse, tr_r2 = metrics(ytr, ytr_pred)
    te_mae, te_rmse, te_r2 = metrics(yte, yte_pred)
    print(f"\n{name}")
    print(" Train  MAE:", np.round(tr_mae, 4), " RMSE:", np.round(tr_rmse, 4), " R2:", np.round(tr_r2, 4))
    print(" Test   MAE:", np.round(te_mae, 4), " RMSE:", np.round(te_rmse, 4), " R2:", np.round(te_r2, 4))
    return yte_pred

# 1. Ridge (baseline linear model)
ridge_pipe = Pipeline([
    ('prep', preprocessor),
    ('reg', MultiOutputRegressor(Ridge(alpha=1.0, random_state=42)))
])
ridge_pipe.fit(X_train, y_train)
y_pred_ridge = eval_model("Ridge (multi-output)", ridge_pipe, X_train, X_test, y_train, y_test)

# 2. RandomForest (multi-output)
rf_pipe = Pipeline([
    ('prep', 'passthrough'),
    ('reg', MultiOutputRegressor(RandomForestRegressor(n_estimators=500, random_state=42, n_jobs=-1)))
])
rf_pipe.fit(X_train, y_train)
y_pred_rf = eval_model("RandomForest (multi-output)", rf_pipe, X_train, X_test, y_train, y_test)

# 3. GradientBoosting (multi-output)
gbr_pipe = Pipeline([
    ('prep', 'passthrough'),
    ('reg', MultiOutputRegressor(GradientBoostingRegressor(n_estimators=600, learning_rate=0.03, random_state=42)))
])
gbr_pipe.fit(X_train, y_train)
y_pred_gbr = eval_model("GradientBoosting (multi-output)", gbr_pipe, X_train, X_test, y_train, y_test)

In [ ]:
# Step 6: Model Diagnostics and Feature Importance

import matplotlib.pyplot as plt
import numpy as np

best_pipe = gbr_pipe  # Gradient Boosting model selected
y_pred = best_pipe.predict(X_test)

# Predicted vs Actual plots for LAS and COD
fig, axs = plt.subplots(1, 2, figsize=(12, 5))
for i, col in enumerate(['LAS', 'COD']):
    axs[i].scatter(y_test[col], y_pred[:, i], alpha=0.7)
    mn, mx = y_test[col].min(), y_test[col].max()
    axs[i].plot([mn, mx], [mn, mx], 'r--')
    axs[i].set_title(f'{col}: Predicted vs Actual')
    axs[i].set_xlabel('Actual')
    axs[i].set_ylabel('Predicted')
plt.tight_layout()
plt.show()

# Residual histograms for LAS and COD
residuals = y_test.values - y_pred
fig, axs = plt.subplots(1, 2, figsize=(12, 5))
axs[0].hist(residuals[:, 0], bins=20, edgecolor='black')
axs[0].set_title('Residuals - LAS')
axs[1].hist(residuals[:, 1], bins=20, edgecolor='black')
axs[1].set_title('Residuals - COD')
plt.tight_layout()
plt.show()

# Feature Importance for LAS model in Gradient Boosting (first target model)
estimator_las = best_pipe.named_steps['reg'].estimators_[0]
importances = estimator_las.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 6))
plt.title('Feature Importance for LAS Prediction (Gradient Boosting)')
plt.bar(range(len(importances)), importances[indices], align='center')
plt.xticks(range(len(importances)), X_train.columns[indices], rotation=90)
plt.ylabel('Relative Importance')
plt.tight_layout()
plt.show()


In [ ]:
# Step 7: Hyperparameter Tuning (Gradient Boosting, multi-output)

from sklearn.model_selection import GridSearchCV
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np # Import numpy
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingRegressor


param_grid = {
    'reg__estimator__n_estimators': [300, 600, 900],
    'reg__estimator__learning_rate': [0.02, 0.03, 0.05],
    'reg__estimator__max_depth': [2, 3, 4],
    'reg__estimator__subsample': [0.7, 0.9, 1.0],
}

search_pipe = Pipeline([
    ('reg', MultiOutputRegressor(GradientBoostingRegressor(random_state=42)))
])

grid = GridSearchCV(
    search_pipe, param_grid=param_grid,
    scoring='neg_mean_squared_error', cv=3, n_jobs=-1, verbose=2
)

grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)
best_gbr = grid.best_estimator_

# Evaluate best tuned model
def eval_model(name, model, Xtr, Xte, ytr, yte):
    ytr_pred = model.predict(Xtr)
    yte_pred = model.predict(Xte)
    def metrics(y_true, y_pred):
        mae = mean_absolute_error(y_true, y_pred, multioutput='raw_values')
        # Calculate RMSE by taking the square root of the mean squared error
        rmse = np.sqrt(mean_squared_error(y_true, y_pred, multioutput='raw_values'))
        r2 = r2_score(y_true, y_pred, multioutput='raw_values')
        return mae, rmse, r2
    tr_mae, tr_rmse, tr_r2 = metrics(ytr, ytr_pred)
    te_mae, te_rmse, te_r2 = metrics(yte, yte_pred)
    print(f"\n{name}")
    print(" Train  MAE:", np.round(tr_mae, 4), " RMSE:", np.round(tr_rmse, 4), " R2:", np.round(tr_r2, 4))
    print(" Test   MAE:", np.round(te_mae, 4), " RMSE:", np.round(te_rmse, 4), " R2:", np.round(te_r2, 4))
    return yte_pred

_ = eval_model("GradientBoosting (tuned multi-output)", best_gbr, X_train, X_test, y_train, y_test)

In [ ]:
# Step 8.1: Persist tuned model and metadata

import json, joblib, time, platform
from datetime import datetime

best_model = best_gbr   # Use the best_gbr from Step 7
model_path = 'best_las_cod_gbr_tuned.pkl'
joblib.dump(best_model, model_path)

metadata = {
    "model_path": model_path,
    "created_at": datetime.utcnow().isoformat() + "Z",
    "python_version": platform.python_version(),
    "library_versions": {
        "sklearn": __import__('sklearn').__version__,
        "pandas": __import__('pandas').__version__,
        "numpy": __import__('numpy').__version__
    },
    "features": X.columns.tolist(),
    "targets": ["LAS", "COD"],
    "best_params": best_gbr.named_steps['reg'].estimator.get_params(), # Get params from the best estimator
    "train_shape": list(X_train.shape),
    "test_shape": list(X_test.shape)
}
with open('best_las_cod_metadata.json','w') as f:
    json.dump(metadata, f, indent=2)

print("Saved:", model_path, "and best_las_cod_metadata.json")